# G5 — Chronos Horizon-Mismatch Confound Check

**Motivation.** Chronos-t5-small's own source (`chronos.py`,
`BaseChronosPipeline.predict()`) warns whenever `prediction_length >
model.config.prediction_length` (=64 for this checkpoint): *"We recommend
keeping prediction length <= 64. The quality of longer predictions may
degrade since the model is not optimized for it."* Nearly every
Panda-vs-Chronos MAE comparison in this log uses H ≥ 96 — out of spec for
Chronos. Panda's own native horizon is 128, so it is evaluated much closer
to its own comfort zone across the same H range. This is a previously
unflagged candidate confound for essentially every headline Panda-wins-
Chronos-loses result in the log, not just one experiment.

**What this notebook does and does not establish.** This checks the
\emph{direction and size} of the confound on one dataset (Weather, matching
Experiment 8's exact protocol for direct comparability). It is not a full
revalidation of every MAE claim in the log — that would require rerunning
each affected experiment at H≤64, which is a much larger undertaking.
Treat this as the initial dose-response check that decides whether that
larger undertaking is warranted.

**Design logic.** Simply shrinking H makes the task easier for *both*
models (the exact confound B3c's fixed-physical-horizon arm ran into,
Section 14). To separate "task got easier for everyone" from "Chronos
specifically was being pushed out of spec," the primary metric is
**relative skill** (chronos_mae / panda_mae, Section 1.2 convention) rather
than absolute advantage. If both models simply find shorter horizons
easier by similar proportions, relative skill should stay roughly flat as H
shrinks. If Chronos specifically closes the gap once H enters its own
trained range, relative skill should move toward 1 disproportionately
faster than task-ease alone would predict.

**Pre-registered decision rule (fixed before running):**
- Cite Experiment 8's Weather H=96 result as the out-of-spec anchor
  (n=20, Panda=0.6378, Chronos=0.8115, relative skill = 1.272).
- Run fresh: Weather H=64 (Chronos's exact native ceiling, in-spec) and
  H=32 (well within spec, extra margin point), same protocol as Experiment
  8 (21 channels, per-window instance norm, n_windows=20).
- **SUPPORTED**: relative skill at H=64 is ≥15% lower (relatively) than at
  H=96, continuing in the same direction at H=32.
- **NOT SUPPORTED**: relative skill at H=64 is within ~10% of H=96's value.
- **INCONCLUSIVE**: anything between — dose-response reported without a
  categorical claim, matching Experiment 31's convention for this exact
  situation.

**One thing this design does NOT control for and should not claim to:**
even a clean SUPPORTED verdict only shows relative skill moves with H in
the predicted direction — it does not prove the *mechanism* is Chronos's
internal chunking/degradation past 64 specifically, as opposed to some
other H-dependent property of Chronos unrelated to the trained ceiling. A
stronger (not yet designed) version would compare Chronos generating H=64
in one native shot vs. an artificially-forced two-chunk 32+32 generation
at the same total H, to isolate the chunking mechanism directly.

## Cell 1 — PASTE-IN PLACEHOLDER

Needs, from your real harness (`new_experiments.ipynb` / `fixed_experiments.ipynb`):
imports, `panda_model`, `chronos_model`, `CONTEXT_LEN`, `instance_norm_window`,
`load_ts` (needed here, unlike B3b — this uses real Weather data, not a
simulator), `panda_forecast`, `chronos_forecast`, `evaluate`.
Not reconstructed from memory — paste your verbatim source.

In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd
from scipy.stats import wilcoxon, linregress
from scipy.integrate import solve_ivp
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 8
CONTEXT_LEN = 512
PRED_LEN    = 96
DATA_DIR    = './ts_data'  # adjust if needed

Device: cpu


In [2]:
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print('Models loaded.')

Models loaded.


In [3]:
# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    """Raw (C, T) — no global normalisation."""
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

# -------------------------------------------------------
# Inference
# -------------------------------------------------------
def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched — all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

# -------------------------------------------------------
# Core evaluator
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='panda', name_b='chronos'):
    """
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: (context_normed: (C,T), horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
    iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)

    result = {
        'label'         : label,
        'horizon'       : horizon,
        'name_a'        : name_a,
        'name_b'        : name_b,
        f'{name_a}_mae' : np.median(mae_a),
        f'{name_a}_iqr' : iqr_a,
        f'{name_b}_mae' : np.median(mae_b),
        f'{name_b}_iqr' : iqr_b,
        'advantage_mae' : adv,
        'wilcoxon_p'    : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}[±{iqr_a:.4f}]  '
        f'{name_b}={np.median(mae_b):.4f}[±{iqr_b:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Helpers defined.')

Helpers defined.


## Cell 2 — Sanity check: does Chronos actually warn at H=96 in this environment?

The GitHub source confirms the warning exists in the `chronos-forecasting`
package, but not that this specific pipeline object/version triggers it, or
that `chronos_forecast`'s wrapper doesn't suppress logging. This checks
empirically rather than assuming the search result applies verbatim to
your loaded model.

In [4]:
import logging

class _CaptureHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(record.getMessage())

_cap = _CaptureHandler()
logging.getLogger().addHandler(_cap)
logging.getLogger().setLevel(logging.WARNING)

# Try to read the config value directly if reachable -- do not assume the
# attribute path, just attempt a couple of plausible ones and report what's found.
found_cfg = None
for attr_path in ["model.config.prediction_length", "config.prediction_length"]:
    obj = chronos_model
    try:
        for part in attr_path.split("."):
            obj = getattr(obj, part)
        found_cfg = obj
        print(f"chronos_model.{attr_path} = {found_cfg}")
        break
    except AttributeError:
        continue
if found_cfg is None:
    print("Could not introspect prediction_length via the attempted attribute paths -- "
          "inspect chronos_model manually if you want the config value directly. "
          "Proceeding with the warning-capture check regardless.")

# Fire one real call at H=96 (out of spec) and see if anything gets logged.
_dummy_context = None  # placeholder -- replaced by a real Weather window in Cell 4
print("\n[This cell's warning-capture check runs for real once Cell 4's Weather "
      "context is available -- see Cell 4a below.]")

chronos_model.model.config.prediction_length = 64

[This cell's warning-capture check runs for real once Cell 4's Weather context is available -- see Cell 4a below.]


## Cell 3 — Cited anchor (Experiment 8, NOT rerun)

In [5]:
import numpy as np
import pandas as pd

# Cited verbatim from the log (Section 4, Experiment 8), n_windows=20, Weather, H=96.
# NOT recomputed here -- avoids a redundant rerun of an already-logged, confirmatory result.
cited_exp8_weather_h96 = {
    "H": 96, "panda_mae": 0.6378, "panda_iqr": 0.1723,
    "chronos_mae": 0.8115, "chronos_iqr": 0.2036,
    "advantage_mae": 0.8115 - 0.6378, "wilcoxon_p": 0.000,
    "relative_skill": 0.8115 / 0.6378,
    "source": "Experiment 8 (cited)", "in_spec_for_chronos": False,
}
print(f"Cited H=96 (out-of-spec for Chronos): adv={cited_exp8_weather_h96['advantage_mae']:+.4f}  "
      f"rel_skill={cited_exp8_weather_h96['relative_skill']:.3f}")

Cited H=96 (out-of-spec for Chronos): adv=+0.1737  rel_skill=1.272


## Cell 4 — Fresh runs: Weather H=64 (Chronos in-spec) and H=32 (extra margin)

In [7]:
# Load Weather exactly as in Experiment 8 -- 21 channels, same source file.
# Uses load_ts from the pasted harness (Cell 1). If your load_ts takes a
# different signature than shown here, adjust to match your actual function
# rather than guessing further -- this call mirrors the convention documented
# elsewhere in this project (Weather = Jena 21-channel 10-minute dataset).
weather_data = load_ts("./ts_data/weather.csv")  # (C, T) or (T, C) per your harness convention -- verify orientation
print(f"Weather loaded: shape={weather_data.shape}")

HORIZONS_TO_RUN = [64, 32]
p_col_candidates = ["wilcoxon_p", "wilcoxon_p_mae", "p"]

def get_p(res):
    return next((res[c] for c in p_col_candidates if c in res and pd.notna(res[c])), np.nan)

g5_results = []
for H in HORIZONS_TO_RUN:
    res = evaluate(weather_data, H, n_windows=20, label=f"Weather_H{H}_g5")
    if res is None:
        print(f"  SKIP H={H}: evaluate() returned None")
        continue
    res["H"] = H
    res["source"] = "fresh"
    res["in_spec_for_chronos"] = (H <= 64)
    res["advantage_mae_recomputed"] = res["chronos_mae"] - res["panda_mae"]
    res["relative_skill"] = res["chronos_mae"] / res["panda_mae"] if res["panda_mae"] > 0 else np.nan
    res["wilcoxon_p_resolved"] = get_p(res)
    g5_results.append(res)
    print(f"  H={H} (in-spec={'Y' if H<=64 else 'N'}): panda={res['panda_mae']:.4f}  "
          f"chronos={res['chronos_mae']:.4f}  adv={res['advantage_mae_recomputed']:+.4f}  "
          f"rel_skill={res['relative_skill']:.3f}  p={res['wilcoxon_p_resolved']:.4f}")

df_g5 = pd.DataFrame(g5_results)
df_g5.to_csv("g5_horizon_mismatch_results.csv", index=False)
print("\nSaved g5_horizon_mismatch_results.csv")

Weather loaded: shape=(21, 52696)
  Weather_H64_g5                                      H=  64  panda=0.5623[±0.2172]  chronos=0.7158[±0.2790]  Adv=+0.1535  p=0.009 *
  H=64 (in-spec=Y): panda=0.5623  chronos=0.7158  adv=+0.1535  rel_skill=1.273  p=0.0086
  Weather_H32_g5                                      H=  32  panda=0.4075[±0.2689]  chronos=0.5411[±0.1457]  Adv=+0.1336  p=0.007 *
  H=32 (in-spec=Y): panda=0.4075  chronos=0.5411  adv=+0.1336  rel_skill=1.328  p=0.0068

Saved g5_horizon_mismatch_results.csv


## Cell 4a — Complete the Cell 2 warning-capture check now that real data exists

In [8]:
# Grab one real 512-step Weather context window and fire chronos_forecast at
# H=96 (out-of-spec) to see if the logging module actually captures a warning
# through your specific chronos_forecast wrapper.
_cap.records.clear()
_test_context = weather_data[:, :CONTEXT_LEN] if weather_data.shape[0] < weather_data.shape[1] else weather_data[:CONTEXT_LEN, :]
try:
    _ = chronos_forecast(_test_context, 96)
    matched = [r for r in _cap.records if "prediction length" in r.lower()]
    if matched:
        print("CONFIRMED: warning fired through this environment's chronos_forecast at H=96:")
        print(f"  {matched[0]}")
    else:
        print("No matching warning captured at H=96. Either logging is suppressed by the "
              "wrapper, this pipeline version handles long horizons without warning, or "
              "the warning uses a different logger not attached to root. Does not by "
              "itself confirm or refute the confound -- only that this specific capture "
              "method didn't catch it.")
except Exception as e:
    print(f"chronos_forecast(context, 96) raised: {e}")
    print("If this is the ValueError from limit_prediction_length=True, that itself "
          "confirms the 64-step ceiling is active and enforced, just in strict mode.")

CONFIRMED: warning fired through this environment's chronos_forecast at H=96:
  We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


## Cell 5 — Dose-response table and pre-registered verdict

In [9]:
rows = [cited_exp8_weather_h96] + g5_results
df_dose = pd.DataFrame(rows)[["H", "source", "in_spec_for_chronos", "panda_mae", "chronos_mae",
                                "relative_skill"]].sort_values("H", ascending=False).reset_index(drop=True)
print("Dose-response: relative skill (chronos_mae / panda_mae) vs. horizon")
print("-" * 80)
print(df_dose.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

rel_96 = cited_exp8_weather_h96["relative_skill"]
row_64 = df_dose[df_dose["H"] == 64]
if len(row_64):
    rel_64 = row_64.iloc[0]["relative_skill"]
    pct_change = (rel_64 - rel_96) / rel_96 * 100
    print(f"\nRelative skill: H=96 (out-of-spec) = {rel_96:.3f}  ->  H=64 (in-spec) = {rel_64:.3f}")
    print(f"Relative change: {pct_change:+.1f}%")

    if pct_change <= -15:
        verdict = "SUPPORTED -- relative skill closes meaningfully once Chronos is back in its trained range"
    elif pct_change >= -10:
        verdict = "NOT SUPPORTED -- relative skill roughly stable; horizon-mismatch is not the driver"
    else:
        verdict = "INCONCLUSIVE -- falls between the pre-registered bounds; report dose-response, no categorical claim"
    print(f"\nPre-registered verdict: {verdict}")

    row_32 = df_dose[df_dose["H"] == 32]
    if len(row_32):
        rel_32 = row_32.iloc[0]["relative_skill"]
        print(f"\nExtra margin point, H=32: relative_skill={rel_32:.3f} "
              f"({(rel_32-rel_96)/rel_96*100:+.1f}% vs H=96) -- "
              f"{'continues in the same direction' if (rel_32-rel_96) <= (rel_64-rel_96) else 'does NOT continue the same direction, treat H=64 result cautiously'}")
else:
    print("H=64 row missing -- check Cell 4 ran successfully.")

Dose-response: relative skill (chronos_mae / panda_mae) vs. horizon
--------------------------------------------------------------------------------
 H               source  in_spec_for_chronos  panda_mae  chronos_mae  relative_skill
96 Experiment 8 (cited)                False     0.6378       0.8115          1.2723
64                fresh                 True     0.5623       0.7158          1.2730
32                fresh                 True     0.4075       0.5411          1.3280

Relative skill: H=96 (out-of-spec) = 1.272  ->  H=64 (in-spec) = 1.273
Relative change: +0.0%

Pre-registered verdict: NOT SUPPORTED -- relative skill roughly stable; horizon-mismatch is not the driver

Extra margin point, H=32: relative_skill=1.328 (+4.4% vs H=96) -- does NOT continue the same direction, treat H=64 result cautiously
